# 🗺️ Generación de Mapas — Análisis de Riesgos CDMX

Este notebook genera mapas interactivos (Folium) y estáticos (Matplotlib + GeoPandas)
para visualizar los resultados del análisis de riesgo.

**Mapas interactivos** → `outputs/modelo/maps/`
**Mapas estáticos** → `outputs/maps/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("."))

from src.config import setup_logging, ensure_output_dirs, MODELO_MAPS_DIR, MAPS_DIR
setup_logging()
ensure_output_dirs()

print(f"Mapas interactivos: {MODELO_MAPS_DIR}")
print(f"Mapas estáticos   : {MAPS_DIR}")

## 1. Cargar datos geoespaciales y modelo

In [ ]:
import joblib
from src.config import MODELO_DIR
from src.map_generator import load_geodata, _add_predictions
from src.preprocessing import get_feature_names

# Cargar GeoDataFrame
gdf = load_geodata()
print(f"AGEBs cargadas: {len(gdf)}")
print(f"Columnas: {list(gdf.columns[:10])}...")
print(f"CRS: {gdf.crs}")

# Cargar modelo
model_path = MODELO_DIR / "modelo_final.joblib"
model = joblib.load(model_path)
feature_names = get_feature_names()
print(f"Modelo cargado: {type(model).__name__}")

# Agregar predicciones
gdf = _add_predictions(gdf, model, feature_names)
if "pred_proba" in gdf.columns:
    print(f"Probabilidades — min: {gdf['pred_proba'].min():.3f}, max: {gdf['pred_proba'].max():.3f}")
    print(f"Predicciones positivas: {(gdf['pred_class'] == 1).sum()} / {len(gdf)}")

## 2. Mapas interactivos (Folium)

### 2.1 Mapa de predicción de riesgo

In [ ]:
from src.map_generator import mapa_riesgo_prediccion
from IPython.display import IFrame

path = mapa_riesgo_prediccion(gdf)
if path:
    print(f"Guardado: {path}")
    display(IFrame(src=str(path), width=800, height=500))

### 2.2 Mapa de probabilidades

In [ ]:
from src.map_generator import mapa_probabilidad

path = mapa_probabilidad(gdf)
if path:
    print(f"Guardado: {path}")
    display(IFrame(src=str(path), width=800, height=500))

### 2.3 Mapa de importancia espacial (feature)

In [ ]:
from src.map_generator import mapa_importancia_espacial

# Mapa del indice de riesgo compuesto
path = mapa_importancia_espacial(gdf, "indice_riesgo_compuesto")
if path:
    print(f"Guardado: {path}")
    display(IFrame(src=str(path), width=800, height=500))

### 2.4 Mapa comparativo multicapa

In [ ]:
from src.map_generator import mapa_comparativo

path = mapa_comparativo(gdf)
if path:
    print(f"Guardado: {path}")
    display(IFrame(src=str(path), width=800, height=500))

## 3. Mapas estáticos (Matplotlib)

### 3.1 Segmentación de riesgo

In [ ]:
from src.map_generator import mapa_segmentacion
import matplotlib.pyplot as plt

path = mapa_segmentacion(gdf)
if path:
    print(f"Guardado: {path}")
    from IPython.display import Image
    display(Image(filename=str(path), width=700))

### 3.2 Grid de múltiples riesgos

In [ ]:
from src.map_generator import mapa_grid_multiple

path = mapa_grid_multiple(gdf)
if path:
    print(f"Guardado: {path}")
    display(Image(filename=str(path), width=800))

### 3.3 Hexbin heatmap

In [ ]:
from src.map_generator import mapa_hexbin_heatmap

path = mapa_hexbin_heatmap(gdf)
if path:
    print(f"Guardado: {path}")
    display(Image(filename=str(path), width=700))

### 3.4 Mapa de errores (FP/FN)

In [ ]:
from src.map_generator import mapa_errores

path = mapa_errores(gdf)
if path:
    print(f"Guardado: {path}")
    display(Image(filename=str(path), width=700))
else:
    print("Se requieren columnas pred_class y alto_riesgo para este mapa.")

### 3.5 Mapa de incertidumbre

In [ ]:
from src.map_generator import mapa_incertidumbre

path = mapa_incertidumbre(gdf)
if path:
    print(f"Guardado: {path}")
    display(Image(filename=str(path), width=700))

## 4. Generar todos los mapas de una vez

In [ ]:
from src.map_generator import generate_all_maps

# Regenerar todos
all_paths = generate_all_maps(model=model, feature_names=feature_names)

print("
" + "=" * 50)
print("  RESUMEN DE MAPAS GENERADOS")
print("=" * 50)
for name, p in all_paths.items():
    status = "ok" if p else "omitido"
    print(f"  [{status:>7}] {name}: {p or 'N/A'}")

---
Mapas interactivos disponibles en  (abrir en navegador).
Mapas estaticos disponibles en .